# 01 - Extract

**Stage:** Extract (the *E* in ETL).

Responsibilities of this notebook:
- Load the **immutable raw** source data from `data/raw/`.
- Perform *only* lightweight validation (shape, columns, dtypes).
- Do **not** clean or transform here — that belongs in `02_transform`.

> Treat `data/raw/` as read-only. Never overwrite source files.

In [3]:
import os

path = os.getcwd()
path = os.path.abspath(os.path.join(path, "..", "data"))
path


'/home/marcel/Documents/github/property-near-the-beach-predictor/data'

## Load raw data

In [4]:
from pathlib import Path
import pandas as pd

RAW_DIR      = Path(path) / "raw"
BEACH_DIR    = RAW_DIR / "beach"
NOT_BEACH_DIR = RAW_DIR / "not_beach"
INTERIM_DIR  = Path(path) / "interim"
INTERIM_DIR.mkdir(parents=True, exist_ok=True)
PROJECT_ROOT = Path(path).parent

records = []
for class_name, label, folder in [("beach", 1, BEACH_DIR), ("not_beach", 0, NOT_BEACH_DIR)]:
    for fp in sorted(folder.rglob("*.jpg")):
        records.append({
            "filepath":   str(fp.relative_to(PROJECT_ROOT)),
            "filename":   fp.name,
            "label":      label,
            "class_name": class_name,
        })

df_raw = pd.DataFrame(records)
print(f"Total images found: {len(df_raw)}")
df_raw.head()


Total images found: 7966


,filepath,filename,label,class_name
0,data/raw/beach/i0001.jpg,i0001.jpg,1,beach
1,data/raw/beach/i0002.jpg,i0002.jpg,1,beach
2,data/raw/beach/i0003.jpg,i0003.jpg,1,beach
3,data/raw/beach/i0004.jpg,i0004.jpg,1,beach
4,data/raw/beach/i0005.jpg,i0005.jpg,1,beach


## Validation & profiling

In [5]:
# Assert both classes are present
assert not df_raw[df_raw["class_name"] == "beach"].empty, "No beach images found in data/raw/beach/"
assert not df_raw[df_raw["class_name"] == "not_beach"].empty, "No not_beach images found in data/raw/not_beach/"

# Verify all files exist on disk
missing = [fp for fp in df_raw["filepath"] if not (PROJECT_ROOT / fp).exists()]
assert not missing, f"{len(missing)} file(s) not found on disk: {missing[:5]}"

print("✓ Both classes present")
print("✓ All files readable\n")
print("Class counts:")
print(df_raw["class_name"].value_counts().to_string())


✓ Both classes present
✓ All files readable

Class counts:
class_name
not_beach    5249
beach        2717


In [6]:
# Random sample from the manifest
df_raw.sample(min(10, len(df_raw)), random_state=42)


,filepath,filename,label,class_name
2716,data/raw/beach/s1020.jpg,s1020.jpg,1,beach
7837,data/raw/not_beach/living_875.jpg,living_875.jpg,0,not_beach
7947,data/raw/not_beach/living_981.jpg,living_981.jpg,0,not_beach
6480,data/raw/not_beach/kitchen_729.jpg,kitchen_729.jpg,0,not_beach
3334,data/raw/not_beach/bed_1009.jpg,bed_1009.jpg,0,not_beach
5072,data/raw/not_beach/din_175.jpg,din_175.jpg,0,not_beach
2287,data/raw/beach/s0529.jpg,s0529.jpg,1,beach
736,data/raw/beach/k0266.jpg,k0266.jpg,1,beach
6574,data/raw/not_beach/kitchen_840.jpg,kitchen_840.jpg,0,not_beach
7614,data/raw/not_beach/living_646.jpg,living_646.jpg,0,not_beach


## Save manifest

In [7]:
MANIFEST_FILE = INTERIM_DIR / "image_manifest.parquet"
df_raw.to_parquet(MANIFEST_FILE, index=False)
print(f"Saved manifest → {MANIFEST_FILE}")
print(f"Rows: {len(df_raw)}  |  Columns: {list(df_raw.columns)}")


Saved manifest → /home/marcel/Documents/github/property-near-the-beach-predictor/data/interim/image_manifest.parquet
Rows: 7966  |  Columns: ['filepath', 'filename', 'label', 'class_name']
